In [1]:
import numpy             as     np
import pandas            as     pd
import geopandas         as     gpd
import os
import time
from   osgeo             import gdal
from   scipy.spatial     import cKDTree
from   joblib            import dump, load


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\jepir\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\jepir\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\jepir\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\Users\jepir\anaconda3\Lib\site-pack

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



In [2]:
# Spectral Indices
def Safe_Division(Numerator, Denominator)                     : return np.where((Denominator == 0) | np.isnan(Denominator), 0, Numerator/Denominator)

# Normalised Difference Indices
def Normalised_Diference(Band_A, Band_B)                      : return Safe_Division(Band_A - Band_B, Band_A + Band_B)

# Normalized Difference Using 3 Bands
def Normalised_Diference_3Bands(Band_A, Band_B, Band_C)       : return Safe_Division(Band_A - (Band_B + Band_C), Band_A + (Band_B + Band_C))

# Other Indices
def ATSAVI(Band_04, Band_08)                                  : return 1.22*(Band_08-1.22*Band_04-0.03)/(1.22*Band_08+Band_04-1.22*0.03+0.08*(1+1.22**2))
def EVI(Band_02, Band_04, Band_08)                            : return 2.5*(Band_08-Band_04)/((Band_08+6*Band_04-7.5*Band_02)+1)
def TNDVI(Band_04, Band_08)                                   : return np.sqrt(Normalised_Diference(Band_08, Band_04)+0.5)
def Pan_NDVI(Band_02, Band_03, Band_04, Band_08)              : return Normalised_Diference(Band_08,(Band_02+Band_03+Band_04))
def Misra_Green_Vegetation(Band_03, Band_04, Band_06, Band_08): return -0.386*Band_03 - 0.530*Band_04 + 0.535*Band_06 + 0.532*Band_08
def Misra_Soil_Brightness(Band_03, Band_04, Band_06, Band_08) : return 0.406*Band_03 + 0.600*Band_04 + 0.645*Band_06 + 0.243*Band_08
def SLAVI(Band_08, Band_11)                                   : return Safe_Division(Band_08, (Band_08 + Band_11))
def MSAVI(Band_08, Band_04)                                   : return (2*Band_08 + 1 - np.sqrt((2*Band_08+1)**2 - 8*(Band_08 - Band_04)))/2
def GSAVI(Band_08, Band_03)                                   : return (0.5 * (Band_08 - Band_03))/(Band_08 + Band_03 + 0.5)
def GARI(Band_08, Band_03, Band_02, Band_04)                  :
    return Safe_Division(Band_08 - Band_03 - (Band_02 - Band_04), Band_08 - Band_03 + (Band_02 - Band_04))

# TCT Variables
def TCT_Brightness(Band_02, Band_03, Band_04, Band_08, Band_11, Band_12):
    return (0.3037*Band_02 + 0.2703*Band_03 + 0.4743*Band_04 +
            0.5585*Band_08 + 0.5082*Band_11 + 0.1863*Band_12)
    
def TCT_Greenness(Band_02, Band_03, Band_04, Band_08, Band_11, Band_12):
    return (-0.2848*Band_02 - 0.2435*Band_03 - 0.5436*Band_04 +
             0.7243*Band_08 + 0.0840*Band_11 - 0.1800*Band_12)
    
def TCT_Wetness(Band_02, Band_03, Band_04, Band_08, Band_11, Band_12):
    return (0.1509*Band_02 + 0.1973*Band_03 + 0.3279*Band_04 +
            0.3406*Band_08 + 0.7112*Band_11 + 0.4572*Band_12)
    
def TCT_Distance(Brightness, Greenness):
    return np.sqrt(Brightness**2 + Greenness**2)
    
def TCT_Angle(Brightness, Greenness):
    return np.arctan2(Greenness, Brightness)

In [3]:
## CHANGE HERE
## SET BIOMASS POOL: AGBD, BGBD, LBD, OR SBD
## INPUT AND OUTPUT FOLDER, THE OUTPUT FOLDER SHOULD BE ALREADY CREATED
BIOMASS_POOL          = 'SBD'
Root_Directory        = 'E:/1_Data_3_Months_Composite/4_Data/_Process/'
Geopackages_Directory = 'C:/_MyOnTheGo/BLS-BIOMASS/ProjectDevelopment/Analysis/Mapping/1_Geopackages/'
Output_Directory      = 'C:/_MyOnTheGo/BLS-BIOMASS/ProjectDevelopment/Analysis/Mapping/V61/2_Datasets/'
Features_Results_File = 'C:/_MyOnTheGo/BLS-BIOMASS/ProjectDevelopment/Analysis/Calibration/V61/Results/TRADITIONAL/' + BIOMASS_POOL + '_FeaturesAnalysis.joblib'
Number_Top_Features   = 36
## SET FEATURES REQUIRED FOR DERIVATIVES
# Leave empty for embeddings
Extra_Features_Array = ['S2_B02','S2_B03','S2_B04', 'S2_B06', 'S2_B08','S2_B11','S2_TCB','S2_TCG']
## END OF CHANGE

In [4]:
# Get Top Features to be saved 
Features_Importance_Results = load(Features_Results_File)
Features_To_Remove_PD       = Features_Importance_Results['Feature_Removed'].loc[Features_Importance_Results['N_Features'] >= Number_Top_Features]
Features_To_Remove_Array    = [
    Feature[0] for Feature in Features_To_Remove_PD[1:]
    if not Feature[0] in Extra_Features_Array
]
# Get Files Paths
Data_Paths = {
                Area_Directory.split('_')[0]:[
                    Root_Directory + Area_Directory + '/' + Dataset_Directory + '/' + Path 
                    for Dataset_Directory in os.listdir(Root_Directory + Area_Directory)
                    if Dataset_Directory != 'FIELD'
                    for Path in os.listdir(Root_Directory + Area_Directory + '/' + Dataset_Directory)
                    if not any(Feature in Path for Feature in Features_To_Remove_Array) and not Path.endswith('.xml')
                ]
                for Area_Directory in os.listdir(Root_Directory)
             }
# Get Study Ares into vector
Study_Areas = Data_Paths.keys()
# Get Geopackages paths for pool and study area
Geopackages_Paths = {
    Path.split('_')[0]:[
        os.path.join(Geopackages_Directory, Path)
    ]
    for Path in os.listdir(Geopackages_Directory)
    if Path.split('_')[0] in Study_Areas and Path.split('_')[-2] == BIOMASS_POOL
}
# Add Geopackages to respective study area data paths
for Area in Study_Areas:
    Data_Paths[Area].append(Geopackages_Paths[Area][0])

In [5]:
# Count time
Time_Process_Start = time.perf_counter()
# Create Datasets - Using Bicubig and NN interpolation
#Study_Areas        = ['SerraLousa']
Biomass_Pools      = [BIOMASS_POOL]
GeoDataFrames_Dict = dict()
Dataset            = ''
for Area in Study_Areas:
    print(Area)
    # Get the GeoPackage file and load it to a GeoPandas Dataframe
    GeoDataFrames_Dict[Area] = gpd.read_file([array for array in Data_Paths[Area] if array.endswith('.gpkg')][0])
    # Extract point coordinates from GeoPackage points
    GPKG_Coordinates_x = GeoDataFrames_Dict[Area].geometry.x.values
    GPKG_Coordinates_y = GeoDataFrames_Dict[Area].geometry.y.values  
    # Get Raster files paths
    Raster_Paths = [array for array in Data_Paths[Area] if array.endswith('.tif')]
    # Iterate Raster Paths to Apply Bicubic Interpolation
    for Path in Raster_Paths:
        T0               = time.perf_counter()
        # Get Dataset Name
        Prev_Dataset = Dataset
        Dataset      = Path.split('/')[-1].split('_')[2]
        # Load the raster file using GDAL
        Temporary_Raster = gdal.Open(Path)
        Temporary_Band   = Temporary_Raster.GetRasterBand(1)
        Temporary_Data   = Temporary_Band.ReadAsArray()
        Temporary_NoData = Temporary_Band.GetNoDataValue()
        # Get raster geotransform to convert pixel indices to coordinates and respective inverse
        Temporary_GT     = Temporary_Raster.GetGeoTransform()
        Temporary_INV_GT = gdal.InvGeoTransform(Temporary_GT)
        # Compute processing time
        T1                = time.perf_counter()
        Processing_Time   = T1 - T0
        # Convert to h:m:s
        hours, rem       = divmod(Processing_Time, 3600)
        minutes, seconds = divmod(rem, 60)
        print(f"Read Task\t\t: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")         
        # Verify if it is a new dataset and if it is required to compute the nearest indices
        if Dataset != Prev_Dataset:
            #print('Computing NN...')
            # Perform Nearest Neighbour Interpolation (Centroid)
            # Get Columns and Rows
            Temporary_Columns, Temporary_Rows = np.meshgrid(np.arange(Temporary_Raster.RasterXSize), np.arange(Temporary_Raster.RasterYSize), indexing='xy')
            # Convert pixel indices to real-world coordinates
            X_Coordinates = Temporary_GT[0] + Temporary_Columns * Temporary_GT[1] + Temporary_Rows * Temporary_GT[2]
            Y_Coordinates = Temporary_GT[3] + Temporary_Columns * Temporary_GT[4] + Temporary_Rows * Temporary_GT[5]
            # Calculate the Centroid Coordinates
            X_Coordinates = X_Coordinates + Temporary_GT[1] / 2
            Y_Coordinates = Y_Coordinates + Temporary_GT[5] / 2
            # Flatten the Coordinate Arrays
            Temporary_Coordinates = np.column_stack((X_Coordinates.flatten(), Y_Coordinates.flatten()))
            Temporary_Values      = Temporary_Data.flatten()
            # Build a KDTree for efficient nearest neighbor search (on raster coordinates)
            Search_Tree = cKDTree(Temporary_Coordinates)
            # For each point in the GeoPackage, find the closest raster pixel
            GPKG_Coordinates   = np.array(list(zip(GeoDataFrames_Dict[Area].geometry.x, GeoDataFrames_Dict[Area].geometry.y)))
            Distances, Indices = Search_Tree.query(GPKG_Coordinates)
        else:
            Temporary_Values = Temporary_Data.flatten()
        # Close the raster dataset
        Temporary_Raster = None
        # Add the interpolated data to the GeoPandas dataframe, under the biomass poll name
        Pool = Path.split('/')[-1].split('_')[2] + '_' + Path.split('/')[-1].split('_')[3]
        GeoDataFrames_Dict[Area][Pool] = Temporary_Values[Indices]
        # Compute processing time
        Time_Process_Ends = time.perf_counter()
        Processing_Time   = Time_Process_Ends - T0
        # Convert to h:m:s
        hours, rem       = divmod(Processing_Time, 3600)
        minutes, seconds = divmod(rem, 60)
        print(f"Pool time\t\t: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")   
        # Add to Biomass Pool if it was not added before
        print(Pool)
        print('-+-+-+-+-+-+-')

# Compute processing time
Time_Process_Ends = time.perf_counter()
Processing_Time   = Time_Process_Ends - Time_Process_Start
# Convert to h:m:s
hours, rem       = divmod(Processing_Time, 3600)
minutes, seconds = divmod(rem, 60)
print(f"Elapsed time: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")

Leon
Read Task		: 00:00:06.66
Pool time		: 00:03:10.97
ALOS2_GLCMSIGMAHVVARIANCE
-+-+-+-+-+-+-
Read Task		: 00:00:09.48
Pool time		: 00:04:19.56
S1GRD_DEM
-+-+-+-+-+-+-
Read Task		: 00:00:08.44
Pool time		: 00:03:32.36
S1SLC_COHVH
-+-+-+-+-+-+-
Read Task		: 00:00:08.28
Pool time		: 00:00:08.76
S1SLC_COHVV
-+-+-+-+-+-+-
Read Task		: 00:00:07.59
Pool time		: 00:00:08.21
S1SLC_GLCMCOHVHCORRELATION
-+-+-+-+-+-+-
Read Task		: 00:00:07.21
Pool time		: 00:00:07.83
S1SLC_GLCMCOHVHVARIANCE
-+-+-+-+-+-+-
Read Task		: 00:00:07.44
Pool time		: 00:00:08.02
S1SLC_GLCMCOHVVCORRELATION
-+-+-+-+-+-+-
Read Task		: 00:00:07.27
Pool time		: 00:00:07.75
S1SLC_PHASEVV
-+-+-+-+-+-+-
Read Task		: 00:00:01.88
Pool time		: 00:01:10.53
S2_B02
-+-+-+-+-+-+-
Read Task		: 00:00:03.47
Pool time		: 00:00:03.63
S2_B03
-+-+-+-+-+-+-
Read Task		: 00:00:02.83
Pool time		: 00:00:02.98
S2_B04
-+-+-+-+-+-+-
Read Task		: 00:00:01.63
Pool time		: 00:00:01.74
S2_B05
-+-+-+-+-+-+-
Read Task		: 00:00:02.42
Pool time		: 00:00:02.

In [6]:
# Count time
Time_Process_Start = time.perf_counter()

# Clean NaN and -9999 values & compute derivates
for Key in GeoDataFrames_Dict.keys():
    # Count Iteration time
    Time_Iteration_Starts = time.perf_counter()
    
    print(Key)
    print('Cleaning...')
    GeoDataFrames_Dict[Key].replace(-9999, np.nan, inplace=True)
    GeoDataFrames_Dict[Key].dropna(inplace=True)
    print('Computing derivatives...')
    # Compute derivatives
    try:
        GeoDataFrames_Dict[Key]['S1GRD_BETAVV'] = 10 * np.log10(GeoDataFrames_Dict[Key]['S1GRD_BETAVV'].clip(lower=1e-12))
    except:
        pass
    try:
        GeoDataFrames_Dict[Key]['S1GRD_BETAVH'] = 10 * np.log10(GeoDataFrames_Dict[Key]['S1GRD_BETAVH'].clip(lower=1e-12))
    except:
        pass
    try:
        GeoDataFrames_Dict[Key]['S1GRD_GAMMAVV'] = 10 * np.log10(GeoDataFrames_Dict[Key]['S1GRD_GAMMAVV'].clip(lower=1e-12))
    except:
        pass
    try:
        GeoDataFrames_Dict[Key]['S1GRD_GAMMAVH'] = 10 * np.log10(GeoDataFrames_Dict[Key]['S1GRD_GAMMAVH'].clip(lower=1e-12))
    except:
        pass
    try:
        GeoDataFrames_Dict[Key]['S1GRD_SIGMAVV'] = 10 * np.log10(GeoDataFrames_Dict[Key]['S1GRD_SIGMAVV'].clip(lower=1e-12))
    except:
        pass
    try:
        GeoDataFrames_Dict[Key]['S1GRD_SIGMAVH'] = 10 * np.log10(GeoDataFrames_Dict[Key]['S1GRD_SIGMAVH'].clip(lower=1e-12))
    except:
        pass
    try:
        GeoDataFrames_Dict[Key]['S1GRD_BETAND'] = (GeoDataFrames_Dict[Key]['S1GRD_BETAVH'] - GeoDataFrames_Dict[Key]['S1GRD_BETAVV']) / (GeoDataFrames_Dict[Key]['S1GRD_BETAVH'] + GeoDataFrames_Dict[Key]['S1GRD_BETAVV'])
    except:
        pass
    try:
        GeoDataFrames_Dict[Key]['S1GRD_GAMMAND'] = (GeoDataFrames_Dict[Key]['S1GRD_GAMMAVH'] - GeoDataFrames_Dict[Key]['S1GRD_GAMMAVV']) / (GeoDataFrames_Dict[Key]['S1GRD_GAMMAVH'] + GeoDataFrames_Dict[Key]['S1GRD_GAMMAVV'])
    except:
        pass
    try:
        GeoDataFrames_Dict[Key]['S1GRD_SIGMAND'] = (GeoDataFrames_Dict[Key]['S1GRD_SIGMAVH'] - GeoDataFrames_Dict[Key]['S1GRD_SIGMAVV']) / (GeoDataFrames_Dict[Key]['S1GRD_SIGMAVH'] + GeoDataFrames_Dict[Key]['S1GRD_SIGMAVV'])
    except:
        pass
    try:
        GeoDataFrames_Dict[Key]['S1GRD_BETARATIO'] = GeoDataFrames_Dict[Key]['S1GRD_BETAVH'] / GeoDataFrames_Dict[Key]['S1GRD_BETAVV']
    except:
        pass
    try:
        GeoDataFrames_Dict[Key]['S1GRD_GAMMARATIO'] = GeoDataFrames_Dict[Key]['S1GRD_GAMMAVH'] / GeoDataFrames_Dict[Key]['S1GRD_GAMMAVV']
    except:
        pass
    try:
        GeoDataFrames_Dict[Key]['S1GRD_SIGMARATIO'] = GeoDataFrames_Dict[Key]['S1GRD_SIGMAVH'] / GeoDataFrames_Dict[Key]['S1GRD_SIGMAVV']
    except:
        pass
    try:
        GeoDataFrames_Dict[Key]['ALOS2_BETAHH'] = 10 * np.log10(GeoDataFrames_Dict[Key]['ALOS2_BETAHH']**2) - 83
    except:
        pass
    try:
        GeoDataFrames_Dict[Key]['ALOS2_BETAHV'] = 10 * np.log10(GeoDataFrames_Dict[Key]['ALOS2_BETAHV']**2) - 83
    except:
        pass
    try:
        GeoDataFrames_Dict[Key]['ALOS2_GAMMAHH'] = 10 * np.log10(GeoDataFrames_Dict[Key]['ALOS2_GAMMAHH']**2) - 83
    except:
        pass
    try:
        GeoDataFrames_Dict[Key]['ALOS2_GAMMAHV'] = 10 * np.log10(GeoDataFrames_Dict[Key]['ALOS2_GAMMAHV']**2) - 83
    except:
        pass
    try:
        GeoDataFrames_Dict[Key]['ALOS2_SIGMAHH'] = 10 * np.log10(GeoDataFrames_Dict[Key]['ALOS2_SIGMAHH']**2) - 83
    except:
        pass
    try:
        GeoDataFrames_Dict[Key]['ALOS2_SIGMAHV'] = 10 * np.log10(GeoDataFrames_Dict[Key]['ALOS2_SIGMAHV']**2) - 83
    except:
        pass
    try:
        GeoDataFrames_Dict[Key]['S2_NDMIB11'] = Normalised_Diference(GeoDataFrames_Dict[Key]['S2_B08'], GeoDataFrames_Dict[Key]['S2_B11'])
    except:
        pass
    try:
        GeoDataFrames_Dict[Key]['S2_NGRDI'] = Normalised_Diference(GeoDataFrames_Dict[Key]['S2_B03'], GeoDataFrames_Dict[Key]['S2_B04'])
    except:
        pass
    try:
        GeoDataFrames_Dict[Key]['S2_NDB08B03B02'] = Normalised_Diference_3Bands(GeoDataFrames_Dict[Key]['S2_B08'], GeoDataFrames_Dict[Key]['S2_B03'], GeoDataFrames_Dict[Key]['S2_B02'])
    except:
        pass
    try:
        GeoDataFrames_Dict[Key]['S2_NDB08B04B03'] = Normalised_Diference_3Bands(GeoDataFrames_Dict[Key]['S2_B08'], GeoDataFrames_Dict[Key]['S2_B04'], GeoDataFrames_Dict[Key]['S2_B03'])
    except:
        pass
    try:
        GeoDataFrames_Dict[Key]['S2_NDB08B04B02'] = Normalised_Diference_3Bands(GeoDataFrames_Dict[Key]['S2_B08'], GeoDataFrames_Dict[Key]['S2_B04'], GeoDataFrames_Dict[Key]['S2_B02'])
    except:
        pass
    try:
        GeoDataFrames_Dict[Key]['S2_MISRAGREENVEGETATION'] = Misra_Green_Vegetation(GeoDataFrames_Dict[Key]['S2_B02'], GeoDataFrames_Dict[Key]['S2_B04'], GeoDataFrames_Dict[Key]['S2_B06'], GeoDataFrames_Dict[Key]['S2_B08'])
    except:
        pass
    try:
        GeoDataFrames_Dict[Key]['S2_MISRAGSOILBRIGHTNESS'] = Misra_Soil_Brightness(GeoDataFrames_Dict[Key]['S2_B02'], GeoDataFrames_Dict[Key]['S2_B04'], GeoDataFrames_Dict[Key]['S2_B06'], GeoDataFrames_Dict[Key]['S2_B08'])
    except:
        pass
    try:
        GeoDataFrames_Dict[Key]['S2_ATSAVI'] = ATSAVI(GeoDataFrames_Dict[Key]['S2_B04'], GeoDataFrames_Dict[Key]['S2_B08'])
    except:
        pass
    try:
        GeoDataFrames_Dict[Key]['S2_TNDVI'] = TNDVI(GeoDataFrames_Dict[Key]['S2_B04'], GeoDataFrames_Dict[Key]['S2_B08'])
    except:
        pass
    try:
        GeoDataFrames_Dict[Key]['S2_PANNDVI'] = Pan_NDVI(GeoDataFrames_Dict[Key]['S2_B02'], GeoDataFrames_Dict[Key]['S2_B03'], GeoDataFrames_Dict[Key]['S2_B04'], GeoDataFrames_Dict[Key]['S2_B08'])
    except:
        pass
    try:
        GeoDataFrames_Dict[Key]['S2_SLAVI'] = SLAVI(GeoDataFrames_Dict[Key]['S2_B08'], GeoDataFrames_Dict[Key]['S2_B11'])
    except:
        pass
    try:
        GeoDataFrames_Dict[Key]['S2_MSAVI'] = MSAVI(GeoDataFrames_Dict[Key]['S2_B08'], GeoDataFrames_Dict[Key]['S2_B04'])
    except:
        pass
    try:
        GeoDataFrames_Dict[Key]['S2_GSAVI'] = GSAVI(GeoDataFrames_Dict[Key]['S2_B08'], GeoDataFrames_Dict[Key]['S2_B03'])
    except:
        pass
    try:
        GeoDataFrames_Dict[Key]['S2_GARI'] = GARI(GeoDataFrames_Dict[Key]['S2_B08'], GeoDataFrames_Dict[Key]['S2_B03'], GeoDataFrames_Dict[Key]['S2_B02'], GeoDataFrames_Dict[Key]['S2_B04'])
    except:
        pass
    try:
        GeoDataFrames_Dict[Key]['S2_TCTDISTANCE'] = TCT_Distance(GeoDataFrames_Dict[Key]['S2_TCB'], GeoDataFrames_Dict[Key]['S2_TCG'])
    except:
        pass
    try:
        GeoDataFrames_Dict[Key]['S2_TCTANGLE'] = TCT_Angle(GeoDataFrames_Dict[Key]['S2_TCB'], GeoDataFrames_Dict[Key]['S2_TCG'])
    except:
        pass
    # Remove non-required features
    print('Removing non-required features...')
    Features_Importance_Results = load(Features_Results_File)
    Features_To_Remove_PD       = Features_Importance_Results['Feature_Removed'].loc[Features_Importance_Results['N_Features'] >= Number_Top_Features]
    Features_To_Remove_Array    = [Feature[0] for Feature in Features_To_Remove_PD[1:]]
    GeoDataFrames_Dict[Key]     = GeoDataFrames_Dict[Key].drop(columns=Features_To_Remove_Array, errors='ignore')
    # Saving geopandas dataframe
    print('Saving Geopandas dataframe...')
    GeoDataFrames_Dict[Key].to_file(Output_Directory + Key + '_' + BIOMASS_POOL + '_FieldDataset.gpkg', driver="GPKG")
    print('Study area concluded.')

    # Compute Iterartion time
    Time_Iteration_Ends = time.perf_counter()
    Iteration_Time   = Time_Iteration_Ends - Time_Iteration_Starts
    # Convert to h:m:s
    hours, rem       = divmod(Iteration_Time, 3600)
    minutes, seconds = divmod(rem, 60)
    print(f"Iteration time\t\t: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")   
    print('-+-+-+-+-+-+-')

# Compute processing time
Time_Process_Ends = time.perf_counter()
Processing_Time   = Time_Process_Ends - Time_Process_Start
# Convert to h:m:s
hours, rem       = divmod(Processing_Time, 3600)
minutes, seconds = divmod(rem, 60)
print(f"Elapsed time: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")

Leon
Cleaning...
Computing derivatives...
Removing non-required features...
Saving Geopandas dataframe...
Study area concluded.
Iteration time		: 00:10:58.13
-+-+-+-+-+-+-
Elapsed time: 00:10:58.13
